# STAGE 12 — FINAL PRODUCTION MODEL TRAINING

**Malaysia-only FYP — Static Random Forest and semi-dynamic Random Forest**

This notebook trains final candidate pipelines on all eligible rows after Stage 11 model selection. Stage 08/09 nested spatial CV metrics remain the held-out estimates. Training metrics are diagnostic only.

**Important:** This stage does not establish nationwide raster coverage or prove temporal causality. Those require separate audits before release.

# PART A — CONFIGURATION

## 1. Libraries, paths and settings

In [ ]:
from pathlib import Path
import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score

RANDOM_SEED = 42
RUN_STATIC = True
RUN_TEMPORAL = True
N_SEARCH_ITER = 20
INNER_SPLITS = 3

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

STATIC_PATH = (
    DEV_DIR
    / "04_model_ready"
    / "static_model_1to2.csv"
)

TEMPORAL_PATH = (
    DEV_DIR
    / "04_model_ready"
    / "temporal_model_full_enhanced_spatial.csv"
)

WATERWAYS_ROOT = (
    RAW_DIR
    / "03_Factors"
    / "Waterways"
)

OUTPUT_DIR = (
    DEV_DIR
    / "05_model_results"
    / "stage12_production_models"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Static input:", STATIC_PATH)
print("Temporal input:", TEMPORAL_PATH)
print("Output:", OUTPUT_DIR)


## 2. Verify input files and class distributions

In [ ]:
if RUN_STATIC and not STATIC_PATH.exists():
    raise FileNotFoundError(STATIC_PATH)

if RUN_TEMPORAL and not TEMPORAL_PATH.exists():
    raise FileNotFoundError(TEMPORAL_PATH)

static = pd.read_csv(STATIC_PATH) if RUN_STATIC else None
temporal = pd.read_csv(TEMPORAL_PATH) if RUN_TEMPORAL else None

if RUN_STATIC:
    if "target" not in static.columns or "spatial_block_id" not in static.columns:
        raise ValueError("Static input needs target and spatial_block_id")

    if len(static) != 582 or int(static["target"].sum()) != 194:
        raise ValueError("Unexpected static cohort: expected 582 rows / 194 cases")

    print("Static:", len(static), "| cases:", int(static["target"].sum()))

if RUN_TEMPORAL:
    if "validation_group" not in temporal.columns:
        raise ValueError("Temporal input needs validation_group")

    if "temporal_role" not in temporal.columns:
        raise ValueError("Temporal input needs temporal_role")

    temporal["target"] = temporal["temporal_role"].eq("case").astype(int)

    if len(temporal) != 363 or int(temporal["target"].sum()) != 121:
        raise ValueError("Unexpected temporal cohort: expected 363 rows / 121 cases")

    print("Temporal:", len(temporal), "| cases:", int(temporal["target"].sum()))


# PART B — STATIC FEATURE ENGINEERING

## 3. Reproduce Stage 08 waterway extraction and derived predictors

This uses the same EPSG:3857 distance approach as Stage 08. Do not silently omit waterway features if Stage 08 used them.

In [ ]:
STATIC_BASE_NUMERIC = [
    "elevation_m",
    "slope_deg",
    "aspect_sin",
    "aspect_cos",
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
    "distance_major_fault_m",
]

STATIC_CATEGORICAL = [
    "geology_age",
    "geology_unit_type",
]

if RUN_STATIC:
    if "distance_waterway_current_m" not in static.columns or not static["distance_waterway_current_m"].notna().any():
        waterway_files = [
            p
            for p in WATERWAYS_ROOT.rglob("*")
            if p.suffix.lower() in {".shp", ".gpkg", ".geojson"}
        ] if WATERWAYS_ROOT.exists() else []

        if not waterway_files:
            raise FileNotFoundError("Stage 08 waterway vectors not found: " + str(WATERWAYS_ROOT))

        water_parts = []

        for path in waterway_files:
            try:
                gdf = gpd.read_file(path)

                if len(gdf) == 0 or gdf.crs is None:
                    continue

                gdf = gdf.to_crs("EPSG:4326")
                gdf = gdf.loc[
                    gdf.geometry.geom_type.isin(["LineString", "MultiLineString"])
                ][["geometry"]].copy()

                if len(gdf):
                    water_parts.append(gdf)

            except Exception as exc:
                print("Skipped:", path, "|", exc)

        if not water_parts:
            raise ValueError("No readable waterway line geometries")

        waterways = gpd.GeoDataFrame(
            pd.concat(water_parts, ignore_index=True),
            geometry="geometry",
            crs="EPSG:4326"
        ).to_crs("EPSG:3857")

        water_union = waterways.geometry.union_all()

        sample_points = gpd.GeoDataFrame(
            static.copy(),
            geometry=gpd.points_from_xy(static["longitude"], static["latitude"]),
            crs="EPSG:4326"
        ).to_crs("EPSG:3857")

        static["distance_waterway_current_m"] = sample_points.geometry.apply(
            lambda point: float(point.distance(water_union))
        ).to_numpy()

    for name in ["distance_major_fault_m", "distance_waterway_current_m"]:
        numeric_values = pd.to_numeric(static[name], errors="coerce")
        if numeric_values.lt(0).any():
            raise ValueError("Negative distance values: " + name)

    static["log_distance_major_fault_m"] = np.log1p(
        pd.to_numeric(static["distance_major_fault_m"], errors="coerce")
    )

    static["slope_sq"] = pd.to_numeric(static["slope_deg"], errors="coerce") ** 2

    static["elevation_slope_interaction"] = (
        pd.to_numeric(static["elevation_m"], errors="coerce")
        * pd.to_numeric(static["slope_deg"], errors="coerce")
    )

    static["log_distance_waterway_current_m"] = np.log1p(
        pd.to_numeric(static["distance_waterway_current_m"], errors="coerce")
    )


## 4. Define and audit predictor schemas

In [ ]:
STATIC_DERIVED = [
    "log_distance_major_fault_m",
    "slope_sq",
    "elevation_slope_interaction",
    "distance_waterway_current_m",
    "log_distance_waterway_current_m",
]

TEMPORAL_NUMERIC = [
    "rain_1d_mm",
    "rain_3d_mm",
    "rain_7d_mm",
    "rain_30d_mm",
    "soil_moisture_0_7cm_7d_mean",
    "soil_moisture_7_28cm_7d_mean",
    "distance_waterway_current_m",
]

TEMPORAL_CATEGORICAL = ["landcover_harmonised"]

branch_configs = {}

if RUN_STATIC:
    required_base = STATIC_BASE_NUMERIC[:]
    missing_base = [c for c in required_base if c not in static.columns]
    if missing_base:
        raise ValueError("Missing required Stage 08 static predictors: " + str(missing_base))

    numeric = required_base + STATIC_DERIVED
    categorical = [
        c for c in STATIC_CATEGORICAL
        if c in static.columns and static[c].notna().any()
    ]

    branch_configs["static"] = {
        "data": static,
        "numeric": numeric,
        "categorical": categorical,
        "group_col": "spatial_block_id",
        "expected_rows": 582,
    }

if RUN_TEMPORAL:
    numeric = [
        c for c in STATIC_BASE_NUMERIC
        if c in temporal.columns and temporal[c].notna().any()
    ] + TEMPORAL_NUMERIC

    categorical = [
        c for c in STATIC_CATEGORICAL
        if c in temporal.columns and temporal[c].notna().any()
    ] + TEMPORAL_CATEGORICAL

    branch_configs["temporal"] = {
        "data": temporal,
        "numeric": list(dict.fromkeys(numeric)),
        "categorical": categorical,
        "group_col": "validation_group",
        "expected_rows": 363,
    }

availability_rows = []

for branch, cfg in branch_configs.items():
    data = cfg["data"]
    features = cfg["numeric"] + cfg["categorical"]

    missing = [c for c in features if c not in data.columns]
    if missing:
        raise ValueError(f"{branch}: missing predictors: {missing}")

    for feature in cfg["numeric"]:
        data[feature] = pd.to_numeric(data[feature], errors="coerce")
        data[feature] = data[feature].replace([np.inf, -np.inf], np.nan)

    all_missing = [c for c in features if data[c].isna().all()]
    if all_missing:
        raise ValueError(f"{branch}: wholly unavailable predictors: {all_missing}")

    for feature in features:
        availability_rows.append({
            "branch": branch,
            "feature": feature,
            "type": "numeric" if feature in cfg["numeric"] else "categorical",
            "nonmissing": int(data[feature].notna().sum()),
            "missing": int(data[feature].isna().sum()),
        })

    print(branch, "| numeric:", cfg["numeric"])
    print(branch, "| categorical:", cfg["categorical"])

pd.DataFrame(availability_rows).to_csv(
    OUTPUT_DIR / "production_feature_availability.csv",
    index=False
)


# PART C — FULL-DEVELOPMENT GROUPED TUNING

## 5. Build a leakage-safe Random Forest pipeline

In [ ]:
RF_PARAM_SPACE = {
    "model__n_estimators": [300, 500, 800],
    "model__max_depth": [3, 5, 7, 9, None],
    "model__min_samples_split": [5, 10, 20, 30],
    "model__min_samples_leaf": [2, 4, 6, 10, 15],
    "model__max_features": ["sqrt", "log2", 0.5],
}


def build_rf_pipeline(numeric_features, categorical_features):
    transformers = [
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_features
        )
    ]

    if categorical_features:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features
        ))

    preprocessing = ColumnTransformer(transformers)

    return Pipeline([
        ("preprocess", preprocessing),
        (
            "model",
            RandomForestClassifier(
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=1
            )
        ),
    ])


## 6. Train and save both final candidate pipelines

Grouped CV is used for hyperparameter selection on the full development dataset. Its score is **not** a new independent validation estimate.

In [ ]:
trained = {}
search_rows = []

for branch, cfg in branch_configs.items():
    print("\nTRAINING:", branch)

    data = cfg["data"]
    features = cfg["numeric"] + cfg["categorical"]
    X = data[features].copy()
    y = data["target"].astype(int).copy()
    groups = data[cfg["group_col"]].astype(str).copy()

    if y.nunique() != 2:
        raise ValueError(branch + ": target must contain both classes")

    group_counts = pd.crosstab(groups, y)
    if group_counts.shape[0] < INNER_SPLITS:
        raise ValueError(branch + ": insufficient validation groups")

    splitter = StratifiedGroupKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    split_indices = list(splitter.split(X, y, groups))

    for train_idx, test_idx in split_indices:
        if len(np.unique(y.iloc[test_idx])) < 2:
            raise ValueError(branch + ": CV fold contains one class; revise groups")

    search = RandomizedSearchCV(
        estimator=build_rf_pipeline(cfg["numeric"], cfg["categorical"]),
        param_distributions=RF_PARAM_SPACE,
        n_iter=N_SEARCH_ITER,
        scoring="roc_auc",
        cv=split_indices,
        refit=True,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbose=0,
        error_score="raise"
    )

    search.fit(X, y)

    best_pipeline = search.best_estimator_
    training_scores = best_pipeline.predict_proba(X)[:, 1]

    model_path = OUTPUT_DIR / f"{branch}_rf_pipeline.joblib"
    joblib.dump(best_pipeline, model_path)

    schema = {
        "branch": branch,
        "algorithm": "Random Forest",
        "numeric_features": cfg["numeric"],
        "categorical_features": cfg["categorical"],
        "feature_order": features,
        "n_rows": int(len(data)),
        "n_cases": int(y.sum()),
        "n_controls": int((y == 0).sum()),
        "spatial_group_column": cfg["group_col"],
        "hyperparameters": search.best_params_,
        "tuning_design": "3-fold StratifiedGroupKFold on all development data",
        "tuning_best_roc_auc": float(search.best_score_),
        "training_roc_auc_diagnostic_only": float(roc_auc_score(y, training_scores)),
        "training_pr_auc_diagnostic_only": float(average_precision_score(y, training_scores)),
        "score_meaning": "relative case-control susceptibility/condition score, NOT event probability",
        "deployment_status": "candidate; national predictor coverage and temporal/leakage audits pending",
    }

    schema_path = OUTPUT_DIR / f"{branch}_feature_schema.json"
    schema_path.write_text(json.dumps(schema, indent=4), encoding="utf-8")

    trained[branch] = {
        "pipeline": best_pipeline,
        "schema": schema,
        "X": X,
        "y": y,
    }

    search_rows.append({
        "branch": branch,
        "rows": len(data),
        "cases": int(y.sum()),
        "controls": int((y == 0).sum()),
        "cv_best_roc_auc_not_independent": search.best_score_,
        "training_roc_auc_diagnostic_only": roc_auc_score(y, training_scores),
        "training_pr_auc_diagnostic_only": average_precision_score(y, training_scores),
        "model_path": str(model_path),
        "schema_path": str(schema_path),
    })

    print("Saved:", model_path)
    print("Saved:", schema_path)

pd.DataFrame(search_rows).to_csv(
    OUTPUT_DIR / "production_training_summary.csv",
    index=False
)


# PART D — DEPLOYMENT READINESS

## 7. Reload the saved model and confirm identical predictions

In [ ]:
verification_rows = []

for branch, item in trained.items():
    model_path = OUTPUT_DIR / f"{branch}_rf_pipeline.joblib"
    restored = joblib.load(model_path)

    X = item["X"]
    saved_scores = item["pipeline"].predict_proba(X)[:, 1]
    restored_scores = restored.predict_proba(X)[:, 1]

    identical = np.allclose(
        saved_scores,
        restored_scores,
        atol=1e-12,
        rtol=1e-12
    )

    if not identical:
        raise AssertionError(branch + ": saved model predictions differ")

    verification_rows.append({
        "branch": branch,
        "loaded_successfully": True,
        "identical_predictions": identical,
        "prediction_min": float(restored_scores.min()),
        "prediction_max": float(restored_scores.max()),
        "n_predictors": len(item["schema"]["feature_order"]),
    })

verification = pd.DataFrame(verification_rows)
verification.to_csv(
    OUTPUT_DIR / "model_reload_verification.csv",
    index=False
)

display(verification)


## 8. Export final readiness checklist

Unfinished audits are deliberately marked as **PENDING** instead of automatically reporting success.

In [ ]:
readiness = pd.DataFrame([
    {
        "check": "Nested spatial model comparison",
        "status": "COMPLETED IN STAGE 11",
        "required_action": "Use Stage 08/09 spatial benchmark results as held-out evidence",
    },
    {
        "check": "Saved pipeline reproducibility",
        "status": "PASSED" if verification["identical_predictions"].all() else "FAILED",
        "required_action": "Confirm joblib pipeline and feature schema",
    },
    {
        "check": "Nationwide feature coverage",
        "status": "PENDING",
        "required_action": "Verify every static predictor can be generated over Peninsular Malaysia, Sabah and Sarawak",
    },
    {
        "check": "Temporal date cutoffs / leakage",
        "status": "PENDING",
        "required_action": "Confirm antecedent data are prior to each event/control date; inspect matched groups",
    },
    {
        "check": "Independent pseudo-absence redraw",
        "status": "NOT COMPLETED",
        "required_action": "Optional further sensitivity or report remaining limitation",
    },
    {
        "check": "Buffered spatial validation",
        "status": "NOT COMPLETED",
        "required_action": "Optional more conservative test or report remaining limitation",
    },
    {
        "check": "Five national susceptibility classes",
        "status": "STAGE 14",
        "required_action": "Assess continuous national scores before setting thresholds",
    },
])

readiness.to_csv(
    OUTPUT_DIR / "production_deployment_readiness.csv",
    index=False
)

display(readiness)


## 9. Outputs to send for verification

Please send the following **small files**:

```text
production_training_summary.csv
production_feature_availability.csv
model_reload_verification.csv
static_feature_schema.json
temporal_feature_schema.json
production_deployment_readiness.csv
```

You do **not** need to upload the `.joblib` pipelines for routine checking. Preserve them locally for Stage 13.

**Stage 13 prerequisites:** nationally continuous elevation/slope/aspect, SoilGrids layers, geology and major-fault coverage, and the national waterway-distance input aligned to the same feature schema. Missing areas must be masked, not filled with invented geological categories.
